# Load dataset

In [0]:
movies_df = spark.read.csv(
    "/Volumes/workspace/default/my_volume_data/movies.csv",
    header= True,
    inferSchema= True
)

display(movies_df)

# Check number of partition distribution

In [0]:
# movies_df.rdd.getNumPartitions() # this can be used in paid version but not with the free version of databricks.

from pyspark.sql.functions import spark_partition_id

movies_with_pid = movies_df.withColumn('partition_id', 
                                       spark_partition_id())

display(movies_with_pid)                                   

In [0]:
# to see how many partitions are being executed: 
movies_with_pid.select("partition_id").distinct().count()

# repartition

In [0]:
repartition_df = movies_df.repartition(5)
display(repartition_df)

In [0]:
repartition_df_pid = repartition_df.withColumn(
    'partition_id', 
    spark_partition_id()
    )

display(repartition_df_pid)


In [0]:
repartition_df_pid.select("partition_id").distinct().count()

In [0]:
repartition_df.explain(True)

#Coalesce

In [0]:
coalesce_df = repartition_df.coalesce(2)

In [0]:
coalesce_df_pid = coalesce_df.withColumn(
    "partition_id", 
    spark_partition_id()
)

display(coalesce_df_pid)

In [0]:
coalesce_df_pid.select('partition_id').distinct().count()

In [0]:
coalesce_df_pid.explain(True)

# real Practical scenairo - Writing files

In [0]:
# writing without Coalesce

repartition_df.write.mode('overwrite').csv('/Volumes/workspace/default/my_volume_data/movies_output_repartitioned')

In [0]:
# writing with Coalesce

coalesce_df.write.mode('overwrite').csv('/Volumes/workspace/default/my_volume_data/movies_output_coalesced')

### This clearly demonstrates why ​coalesce is commonly used before writing data. It helps reduce the small file problem. Too ​many small files can slow down downstream queries, increase metadata overhead, reduce ​overall system efficiency. Since we are using the free edition with limited cores, increasing ​partitions does not automatically improve performance. If your cluster has limited CPU ​cores, too many partitions can create scheduling overhead. Too few partitions can reduce parallelism. 